# Interpretability smoke test

End-to-end demo of the MK classifier audit on a small sample (~500 GES UVES spectra). Runs on a laptop in under 5 minutes.

**Prerequisites:** `pip install -e '.[interpret]'` and a populated `features.npz` produced by `scripts/build_features.py` (or the synthetic fallback below).

Artifacts produced: permutation importance, SHAP values, sliding-window occlusion trace, masked-line ablation against a random-null, and the two summary PDFs.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from src.interpret.lines import ALLOWED_MK_CLASSES, LINE_SETS, MK_LINES
from src.interpret.features import (
    DEFAULT_WAVE_MAX, DEFAULT_WAVE_MIN, rebin_flux, _split_indices,
)
from src.interpret.classifier import train, evaluate
from src.interpret.importance import compute_permutation_importance
from src.interpret.shap_explain import compute_shap_values, mean_abs_shap_per_class
from src.interpret.occlusion import masked_line_ablation, sliding_window_occlusion
from src.interpret.plotting import plot_summary_overlay


## 1. Load or synthesize features

If `data/interpret/features.npz` exists we use it. Otherwise we synthesize 500 spectra with injected class-dependent absorption at `MK_LINES` positions so the rest of the notebook still runs.

In [ ]:
features_path = Path('data/interpret/features.npz')
if features_path.exists():
    payload = dict(np.load(features_path, allow_pickle=False))
    X, y = payload['X'], payload['y'].astype(int)
    wc = payload['wave_centers']
    train_idx = payload['train_idx']
    val_idx = payload['val_idx']
    test_idx = payload['test_idx']
else:
    rng = np.random.default_rng(0)
    n_spec, n_pix = 500, 3600
    wave = np.linspace(DEFAULT_WAVE_MIN, DEFAULT_WAVE_MAX, n_pix)
    y = rng.integers(0, len(ALLOWED_MK_CLASSES), size=n_spec)
    flux = 1.0 + 0.01 * rng.standard_normal((n_spec, n_pix)).astype(np.float32)
    for i, c in enumerate(y):
        cls = ALLOWED_MK_CLASSES[c]
        for line in MK_LINES:
            if cls in line.diag_for:
                depth = 0.25 + 0.05 * rng.standard_normal()
                sigma = 1.5
                flux[i] -= depth * np.exp(-0.5 * ((wave - line.wavelength_aa) / sigma) ** 2)
    X, wc = rebin_flux(flux, wave, rebin_factor=5)
    train_idx, val_idx, test_idx = _split_indices(y.astype(np.int8), None, 0.7, 0.15, seed=0)
    print('synthetic fallback:', X.shape)
print('features shape:', X.shape, 'classes present:', sorted(set(y.tolist())))


## 2. Train LightGBM and evaluate

In [ ]:
present = sorted(np.unique(y).tolist())
class_labels = [ALLOWED_MK_CLASSES[i] for i in present]
model = train(
    X[train_idx], y[train_idx],
    X[val_idx], y[val_idx],
    num_class=len(present),
    hparams={'n_estimators': 100},  # keep the smoke fast
)
metrics = evaluate(model, X[test_idx], y[test_idx], class_labels=class_labels,
                   n_train=len(train_idx), n_val=len(val_idx))
print(f"accuracy={metrics.accuracy:.3f}  macro_f1={metrics.macro_f1:.3f}")


## 3. Permutation importance + SHAP

In [ ]:
imp_mean, imp_std = compute_permutation_importance(
    model, X[val_idx], y[val_idx], n_repeats=3, seed=0,
)
shap_vals = compute_shap_values(model, X[val_idx][:200])
mabs = mean_abs_shap_per_class(shap_vals)
print('top-5 permutation bins (wavelength):', wc[np.argsort(imp_mean)[-5:][::-1]])


## 4. Sliding-window occlusion + masked-line ablation (small bootstrap)

In [ ]:
centers, delta = sliding_window_occlusion(model, X[test_idx], y[test_idx], wc)
rows = masked_line_ablation(
    model, X[test_idx], y[test_idx], wc, LINE_SETS, class_labels,
    per_class=True, n_bootstrap=50, n_random_controls=20, seed=0,
)
for r in rows[:8]:
    print(f"{r.line_set:8s} class={r.mk_class:5s}  delta={r.delta_acc_mean:+.3f}  p={r.p_value_vs_random:.3f}")


## 5. Summary overlay

In [ ]:
rep = np.nanmedian(X, axis=0)
out = Path('figures/smoke_overlay.pdf')
plot_summary_overlay(wc, imp_mean, mabs, class_labels, rep, out)
print('wrote', out)
